# TextVQA basis and stage localization

Separate version: 16 conditions,96→32 after block16. Native-drop preserves original basis replacement/order and disables merge; FPS-only and crossed stages test localization.

Attach the same `rq1-ih-inputs-v1`, enable Internet and T4×2. Run zero-skip prerequisites and five-question calibration first (80 predictions). Review before full100 (1600 predictions). Layer16 states/norms are saved and deduplicated; allow roughly1GiB extra output space.

Do not update code in a running GQA process. Use this notebook after GQA finishes. No OCR boxes or extra input bundle needed. See `research/BASIS_LOCALIZATION.md`.


In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path

RUN_LOCALIZATION_CALIBRATION = False  # Set True to enable setup. Full run is separately gated below.
REPO_URL = "https://github.com/doguineyes/CSE590-ApET.git"
REPO_REF = "apet-analysis"
INPUT_ZIP = None  # Existing rq1-ih-inputs-v1 Dataset/ZIP; not the older rq1-inputs-v1.
CHECKPOINT_ARGS = []  # Optional existing original-format model and vision paths.

if not RUN_LOCALIZATION_CALIBRATION:
    raise RuntimeError("Set RUN_LOCALIZATION_CALIBRATION = True, then run cells manually.")
print("Preparing basis/localization experiment; model inference has not started.", flush=True)
repo = Path("/kaggle/working/CSE590-ApET")
if not repo.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True, timeout=120)
subprocess.run(["git", "fetch", "origin"], cwd=repo, check=True, timeout=120)
subprocess.run(["git", "checkout", REPO_REF], cwd=repo, check=True, timeout=120)
if REPO_REF == "apet-analysis":
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True, timeout=120)
subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo, check=True, timeout=120)
if not (repo / "scripts/run_basis_localization.py").is_file():
    raise RuntimeError("Push the new staged TextVQA100 files first, then update this checkout.")
original_python = repo / "environments/original/.venv/bin/python"
localization_runs = []
print("Repository:", repo, flush=True)


## Existing input bundle and locked runtime

In [ ]:
sys.path.insert(0, str(repo / "scripts"))
from rq1_input_ih import locate_inputs
inputs = locate_inputs(
    [Path("/kaggle/input"), Path("/kaggle/working")],
    Path("/kaggle/working/rq1-ih-inputs-v1"), explicit=INPUT_ZIP,
)
print("Accepted input directory:", inputs, flush=True)
setup_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
setup_log = Path(f"/kaggle/working/rq2-staged-textvqa-setup-{setup_stamp}.log")
runtime_report = Path(f"/kaggle/working/rq2-staged-textvqa-runtime-{setup_stamp}.json")

def stream_command(command, stream, *, cwd=None):
    print("Running:", " ".join(map(str, command)), flush=True)
    stream.write("COMMAND " + repr(command) + "\n")
    stream.flush()
    process = subprocess.Popen(command, cwd=cwd, env={**os.environ, "MPLBACKEND": "Agg"}, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, encoding="utf-8", bufsize=1)
    lines = []
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            stream.write(line)
            stream.flush()
            lines.append(line)
        status = process.wait()
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        process.stdout.close()
    stream.write(f"EXIT {status}\n")
    stream.flush()
    if status:
        raise RuntimeError(f"Command failed (exit {status}); preserve {stream.name} before changing anything.")
    return "".join(lines)

with setup_log.open("x", encoding="utf-8") as stream:
    stream_command([
        "bash", "bootstrap/setup_original_linux.sh", "--require-cuda", "--expected-gpus", "2",
        "--output", str(runtime_report),
    ], stream, cwd=repo)
print("Locked original runtime ready. Run the prerequisite/helper cell next.", flush=True)
print("Setup log:", setup_log)
print("Runtime report:", runtime_report)


## Actual tiny-native tests and helpers

In [ ]:
def verify_localization_prerequisites():
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    proof = Path(f"/kaggle/working/basis-localization-prerequisites-{stamp}.json")
    with setup_log.open("a", encoding="utf-8") as stream:
        stream_command([str(original_python), "-X", "utf8", "-I", "scripts/run_basis_localization.py",
            "--check", "--output", str(proof)], stream, cwd=repo)
    result = json.loads(proof.read_text())
    if result["status"] != "passed" or result["skipped"]:
        raise RuntimeError("Require actual native tests, zero skips")
    print("Prerequisites:", result["tests"], "passed; zero skips")
    return proof


def run_basis_localization(mode="calibration", dataset="textvqa", calibration_report=None):
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    folder = Path(f"/kaggle/working/basis-localization-{dataset}-{mode}-v1-{stamp}")
    localization_runs.append(folder)
    command = [str(original_python), "-X", "utf8", "-I", "scripts/run_basis_localization.py",
        "--mode", mode, "--dataset", dataset, "--output", str(folder),
        "--prerequisite-report", str(localization_prerequisites), "--setup-log", str(setup_log),
        "--runtime-report", str(runtime_report), *CHECKPOINT_ARGS]
    command.extend(["--inputs", str(inputs)])
    if calibration_report is not None: command.extend(["--calibration-report", str(calibration_report)])
    try:
        with folder.with_suffix(".log").open("x", encoding="utf-8") as stream:
            stream_command(command, stream, cwd=repo)
    finally:
        print("Download:", folder.with_suffix(".zip"), folder.with_suffix(".log"), flush=True)
    report = json.loads((folder / "report.json").read_text())
    print({k: report.get(k) for k in ("status", "completed_questions", "completed_predictions", "wall_seconds", "error")})
    return folder

localization_prerequisites = verify_localization_prerequisites()


## Calibration first: download ZIP/log for review

In [ ]:
localization_calibration_dir = run_basis_localization()


## Full TextVQA100 — enable after calibration review

Keep this session to reuse installed packages and model cache. Source changes invalidate
calibration. Results preserve masks, basis/source mappings, scores and pre-prune layer16 states/norms.


In [ ]:
RUN_FULL_TEXTVQA = False
CALIBRATION_REPORT = localization_calibration_dir / "report.json"
if RUN_FULL_TEXTVQA:
    localization_textvqa_dir = run_basis_localization("full", "textvqa", CALIBRATION_REPORT)


## Download before stopping

In [ ]:
from IPython.display import FileLink, display
for folder in localization_runs:
    for p in (folder.with_suffix(".zip"), folder.with_suffix(".log")):
        if p.exists(): display(FileLink(str(p)))
